# LLM Agora Demo
Interactively run a simple Agora session with configurable agents, LLM backends, and turn limits.

## Instructions
- Ensure a `.env` file defines `OPENROUTER_API_KEY`.
- Adjust `agent_configs` / `turns_per_agent` to explore different models and prompts.
- Run the cells sequentially to create agents, execute the Agora, and inspect each agent's view of the shared history.


In [1]:
from dotenv import load_dotenv

load_dotenv()

from agora.agora import Agora
from agora.agent import Agent
from agora.llm import OpenRouterClient


In [2]:
# --- Configuration ---
turns_per_agent = 2
agent_configs = [
    {
        'name': 'Alpha',
        'model': 'openai/gpt-4o-mini',
        'system_prompt': 'You are Alpha, a seller of a widget who speaks in concise, single sentences.',
    },
    {
        'name': 'Beta',
        'model': 'anthropic/claude-3-haiku',
        'system_prompt': 'You are Beta, a skeptical potential widget buyer who speaks in concise, single sentences.',
    },
]


In [3]:
# --- Build agents and run the Agora ---
client = OpenRouterClient()
try:
    agents = []
    for cfg in agent_configs:
        agent = Agent(
            name=cfg['name'],
            model=cfg['model'],
            llm_client=client,
            system_prompt=cfg.get('system_prompt', ''),
        )
        agents.append(agent)

    agora = Agora(agents)
    history = agora.run(max_turns_per_agent=turns_per_agent)

    print(f'Total turns: {len(history)}')
    for turn in history:
        speaker = turn.metadata.get('speaker_name', turn.speaker_id)
        print(f"Turn {turn.turn_id:02d} | {speaker}: {turn.public_speech}")
finally:
    client.close()


Total turns: 4
Turn 01 | Alpha: Discover the best widget on the market today!
Turn 02 | Beta: Prove it.
Turn 03 | Alpha: Our widgets are made with high-quality materials and come with a 100% satisfaction guarantee!
Turn 04 | Beta: I'll need more evidence than that.


In [4]:
# --- Inspect each agent's perspective on the history ---
for agent in agents:
    print(f"\n### History visible to {agent.name}")
    for turn in agent.view_history():
        speaker = turn.metadata.get('speaker_name', turn.speaker_id)
        print(f"Turn {turn.turn_id:02d} | {speaker}: {turn.public_speech}")



### History visible to Alpha
Turn 01 | Alpha: Discover the best widget on the market today!
Turn 02 | Beta: Prove it.
Turn 03 | Alpha: Our widgets are made with high-quality materials and come with a 100% satisfaction guarantee!
Turn 04 | Beta: I'll need more evidence than that.

### History visible to Beta
Turn 01 | Alpha: Discover the best widget on the market today!
Turn 02 | Beta: Prove it.
Turn 03 | Alpha: Our widgets are made with high-quality materials and come with a 100% satisfaction guarantee!
Turn 04 | Beta: I'll need more evidence than that.
